# GNRHR (LHRH receptor) expression across TNBC subtypes

Background: LHRH-conjugated nanoparticle constructs (PEG-coated magnetite; triptorelin-conjugated gold) are an active strategy for targeting triple-negative breast cancer cells by binding the LHRH/GnRH receptor on the tumor cell surface. Published work on these constructs establishes the targeting mechanism but doesn't directly address a follow-up question: **does receptor expression vary across the Lehmann molecular subtypes of TNBC**, which would matter for how uniformly LHRH-nanoparticle constructs could target different TNBC tumors.

Data: TCGA Breast Invasive Carcinoma (Firehose Legacy), downloaded locally from cBioPortal (`brca_tcga/`).

Note on gene target: GNRHR is the receptor gene analyzed here, distinct from **GNRH1**, the gene for the ligand hormone itself. The nanoparticle constructs bind the **receptor**, so GNRHR (not GNRH1) is the relevant target gene throughout this analysis.

In [1]:
import json
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries loaded ✓")

All libraries loaded ✓


## 1. Build the TNBC cohort

TNBC is defined as ER-negative, PR-negative, HER2-negative by IHC, pulled from the local clinical patient file.

In [2]:
STUDY_DIR = "../brca_tcga"

clin_raw = pd.read_csv(f"{STUDY_DIR}/data_clinical_patient.txt", sep="\t", header=None, skiprows=4)
header = clin_raw.iloc[0].tolist()
clin = clin_raw.iloc[1:].copy()
clin.columns = header
clin = clin.reset_index(drop=True)

tnbc_patients = clin[
    (clin["ER_STATUS_BY_IHC"] == "Negative") &
    (clin["PR_STATUS_BY_IHC"] == "Negative") &
    (clin["IHC_HER2"] == "Negative")
][["PATIENT_ID", "OS_STATUS", "OS_MONTHS"]].copy()
tnbc_patients.columns = ["patientId", "OS_STATUS", "OS_MONTHS"]
print(f"TNBC patients (ER-/PR-/HER2-): {len(tnbc_patients)}")

TNBC patients (ER-/PR-/HER2-): 116


## 2. Pull GNRHR + Lehmann marker-gene expression

TCGA's Firehose Legacy BRCA study doesn't ship Lehmann subtype (BL1/BL2/M/MSL/IM/LAR) calls directly, so subtype is approximated from eight established marker genes (two per subtype, z-scored, subtype = highest-scoring pair): CCNE1/CDC6 (BL1), EGFR/MET (BL2), VIM/ZEB1 (M), AR/FOXA1 (LAR). This is a proxy, not the full Lehmann classifier — noted as a limitation below.

In [3]:
genes_needed = ["GNRHR", "CCNE1", "CDC6", "EGFR", "MET", "VIM", "ZEB1", "AR", "FOXA1"]

expr = pd.read_csv(f"{STUDY_DIR}/data_mrna_seq_v2_rsem.txt", sep="\t")
expr_sub = expr[expr["Hugo_Symbol"].isin(genes_needed)].copy()
expr_sub = expr_sub.drop(columns=["Entrez_Gene_Id"]).set_index("Hugo_Symbol")

expr_t = expr_sub.T
expr_t["patientId"] = expr_t.index.str[:12]
expr_t["sampleId"] = expr_t.index
expr_t = expr_t[expr_t["sampleId"].str.endswith("-01")]
expr_t = expr_t.drop_duplicates(subset="patientId")

tnbc_expr = expr_t[expr_t["patientId"].isin(tnbc_patients["patientId"])].copy()
for g in genes_needed:
    tnbc_expr[g] = pd.to_numeric(tnbc_expr[g], errors="coerce")
print(f"TNBC samples with expression data: {len(tnbc_expr)}")

TNBC samples with expression data: 115


In [4]:
marker_genes = ["CCNE1", "CDC6", "EGFR", "MET", "VIM", "ZEB1", "AR", "FOXA1"]
scaler = StandardScaler()
scaled = scaler.fit_transform(tnbc_expr[marker_genes])
scaled_df = pd.DataFrame(scaled, columns=marker_genes, index=tnbc_expr.index)

scaled_df["BL1_score"] = scaled_df[["CCNE1", "CDC6"]].mean(axis=1)
scaled_df["BL2_score"] = scaled_df[["EGFR", "MET"]].mean(axis=1)
scaled_df["M_score"] = scaled_df[["VIM", "ZEB1"]].mean(axis=1)
scaled_df["LAR_score"] = scaled_df[["AR", "FOXA1"]].mean(axis=1)

score_cols = ["BL1_score", "BL2_score", "M_score", "LAR_score"]
tnbc_expr["subtype"] = scaled_df[score_cols].idxmax(axis=1).str.replace("_score", "")
print(tnbc_expr["subtype"].value_counts())

subtype
BL1    44
M      42
LAR    15
BL2    14
Name: count, dtype: int64


## 3. Does GNRHR expression differ across subtypes?

In [5]:
final_df = tnbc_expr.merge(tnbc_patients, on="patientId", how="left")
final_df["GNRHR"] = pd.to_numeric(final_df["GNRHR"], errors="coerce")
final_df["OS_MONTHS"] = pd.to_numeric(final_df["OS_MONTHS"], errors="coerce")

order = ["BL1", "BL2", "M", "LAR"]
groups = [final_df.loc[final_df["subtype"] == s, "GNRHR"].dropna().values for s in order]
kw_stat, kw_p = stats.kruskal(*groups)
print(f"Kruskal-Wallis across subtypes: H={kw_stat:.3f}, p={kw_p:.4f}")
for s, g in zip(order, groups):
    print(f"  {s}: n={len(g)}, median={np.median(g):.2f}, mean={np.mean(g):.2f}")

Kruskal-Wallis across subtypes: H=1.376, p=0.7111
  BL1: n=44, median=2.55, mean=3.22
  BL2: n=14, median=2.89, mean=3.31
  M: n=42, median=3.08, mean=4.23
  LAR: n=15, median=4.55, mean=4.03


In [6]:
from itertools import combinations
pairwise = {}
for a, b in combinations(order, 2):
    ga = final_df.loc[final_df["subtype"] == a, "GNRHR"].dropna().values
    gb = final_df.loc[final_df["subtype"] == b, "GNRHR"].dropna().values
    u, p = stats.mannwhitneyu(ga, gb, alternative="two-sided")
    pairwise[f"{a}_vs_{b}"] = {"U": float(u), "p": float(p)}
    print(f"  {a} vs {b}: U={u:.1f}, p={p:.4f}")

  BL1 vs BL2: U=278.0, p=0.5919
  BL1 vs M: U=805.0, p=0.3059
  BL1 vs LAR: U=286.0, p=0.4489
  BL2 vs M: U=275.0, p=0.7263
  BL2 vs LAR: U=94.0, p=0.6467
  M vs LAR: U=301.0, p=0.8067


In [7]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=final_df, x="subtype", y="GNRHR", order=order, hue="subtype", legend=False, palette="muted", ax=ax, cut=0)
sns.stripplot(data=final_df, x="subtype", y="GNRHR", order=order, color="black", alpha=0.35, size=3, ax=ax)
ax.set_xlabel("TNBC Subtype (Lehmann, marker-gene proxy)", fontsize=12)
ax.set_ylabel("GNRHR Expression (RNA-seq V2 RSEM)", fontsize=12)
ax.set_title("GNRHR (LHRH Receptor) Expression Across TNBC Subtypes", fontsize=13)
ax.text(0.02, 0.98, f"Kruskal-Wallis p = {kw_p:.3f}", transform=ax.transAxes, va="top", fontsize=10, style="italic")
plt.tight_layout()
plt.savefig("../results/gnrhr_by_subtype.png", dpi=150)
plt.show()
print("Saved gnrhr_by_subtype.png")

Saved gnrhr_by_subtype.png


## 4. Does GNRHR expression predict survival?

Median-split TNBC patients into GNRHR-high vs GNRHR-low and compare overall survival.

In [8]:
median_gnrhr = final_df["GNRHR"].median()
final_df["gnrhr_group"] = np.where(final_df["GNRHR"] >= median_gnrhr, "high", "low")

surv_df = final_df.dropna(subset=["OS_MONTHS", "OS_STATUS"]).copy()
surv_df["event"] = surv_df["OS_STATUS"].astype(str).str.startswith("1").astype(int)

os_high = surv_df.loc[surv_df["gnrhr_group"] == "high", "OS_MONTHS"]
os_low = surv_df.loc[surv_df["gnrhr_group"] == "low", "OS_MONTHS"]
os_u, os_p = stats.mannwhitneyu(os_high, os_low, alternative="two-sided")

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

lr = logrank_test(
    surv_df.loc[surv_df["gnrhr_group"] == "high", "OS_MONTHS"],
    surv_df.loc[surv_df["gnrhr_group"] == "low", "OS_MONTHS"],
    event_observed_A=surv_df.loc[surv_df["gnrhr_group"] == "high", "event"],
    event_observed_B=surv_df.loc[surv_df["gnrhr_group"] == "low", "event"],
)
print(f"n evaluable for survival: {len(surv_df)} (high={len(os_high)}, low={len(os_low)})")
print(f"Mann-Whitney on OS months, high vs low GNRHR: p={os_p:.4f}")
print(f"Log-rank test p={lr.p_value:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
kmf = KaplanMeierFitter()
for label, mask in [("GNRHR high", surv_df["gnrhr_group"] == "high"),
                     ("GNRHR low", surv_df["gnrhr_group"] == "low")]:
    kmf.fit(surv_df.loc[mask, "OS_MONTHS"], surv_df.loc[mask, "event"], label=label)
    kmf.plot_survival_function(ax=ax)
ax.set_xlabel("Months")
ax.set_ylabel("Overall survival probability")
ax.set_title(f"TNBC overall survival by GNRHR expression (log-rank p={lr.p_value:.3f})")
plt.tight_layout()
plt.savefig("../results/gnrhr_survival_km.png", dpi=150)
plt.show()
print("Saved gnrhr_survival_km.png")

n evaluable for survival: 115 (high=58, low=57)
Mann-Whitney on OS months, high vs low GNRHR: p=0.1030
Log-rank test p=0.9834
Saved gnrhr_survival_km.png


In [9]:
results = {
    "n_tnbc_patients": int(len(tnbc_patients)),
    "n_tnbc_with_expression": int(len(final_df)),
    "subtype_counts": final_df["subtype"].value_counts().to_dict(),
    "kruskal_wallis": {"H": float(kw_stat), "p": float(kw_p)},
    "pairwise_mannwhitney": pairwise,
    "median_gnrhr_split": float(median_gnrhr),
    "survival_mannwhitney_p": float(os_p),
    "survival_logrank_p": float(lr.p_value),
    "n_survival_evaluable": int(len(surv_df)),
}
final_df.to_csv("../results/gnrhr_tnbc_final.csv", index=False)
with open("../results/analysis_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("results saved to analysis_results.json and gnrhr_tnbc_final.csv")

results saved to analysis_results.json and gnrhr_tnbc_final.csv


## 5. Interpretation

**GNRHR expression does not differ significantly across the four Lehmann-proxy TNBC subtypes** (Kruskal-Wallis H=1.38, p=0.71; all six pairwise comparisons p>0.3). LAR trends numerically highest (median 4.55 vs 2.55-3.08 for the others), consistent with LAR's known androgen/hormone-receptor-driven biology, but the difference isn't statistically significant — likely underpowered given the small LAR/BL2 groups (n=14-15).

**GNRHR expression (high vs low, median split) does not predict overall survival in this cohort** (log-rank p=0.98).

**Summary:** this is a real, defensible null result, not a failed analysis. Receptor transcript availability looks reasonably consistent across TNBC molecular subtypes in this cohort, which suggests LHRH-nanoparticle constructs are not systematically disadvantaged in any one subtype based on receptor availability alone — though the modest LAR/BL2 sample sizes and the fact that subtype here is a marker-gene proxy (not the full Lehmann classifier, which TCGA doesn't provide) both limit how strongly that can be claimed. An open mechanistic question remains: does the receptor's downstream signaling or trafficking differ by subtype even when raw expression doesn't?

**Limitations:**
- Subtype assignment is a two-marker-gene z-score proxy, not the full Lehmann PAM50-derived classifier (not available in this TCGA study).
- TNBC cohort defined by IHC ER/PR/HER2 status only (n=116), not confirmed by additional molecular criteria.
- Small subgroup sizes (BL2 n=14, LAR n=15) limit power to detect moderate effect sizes.